# Feature Engineering — Economic Impact Dashboard

Merge environmental and economic datasets, create lagged features,
and prepare panel data for regression analysis.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

## Feature Engineering Steps

1. Merge environmental and economic datasets on region and date
2. Create lagged features (1, 3, 6 month lags)
3. Compute rolling averages and cumulative anomalies
4. Prepare panel data structure (region x time)
5. Handle missing values and normalize features

In [ ]:
# Load and merge datasets
econ = pd.read_csv('../data/raw/economic_indicators.csv', parse_dates=['date'])
env = pd.read_csv('../data/raw/environmental_variables.csv', parse_dates=['date'])

panel = econ.merge(env, on=['region', 'date'], how='inner')
print(f'Merged panel: {panel.shape}')

In [ ]:
# Create lagged features
env_cols = ['ndvi_mean', 'precipitation', 'temperature_avg']
for col in env_cols:
    for lag in [1, 3, 6]:
        panel[f'{col}_lag{lag}'] = panel.groupby('region')[col].shift(lag)

# Rolling averages
for col in env_cols:
    panel[f'{col}_roll6_mean'] = panel.groupby('region')[col].transform(
        lambda x: x.rolling(6).mean()
    )

# Drop rows with NaN from lagging
panel = panel.dropna()
panel.to_parquet('../data/processed/panel_features.parquet', index=False)
print(f'Panel features: {panel.shape}')
print(f'Columns: {panel.columns.tolist()}')